## **Twitter (X) sentiment analysis using ML.**

In [172]:
# Installing the kaggle library
! pip install kaggle

# Import the zipfile lib to extract data from the zip file.

In [179]:
from zipfile import ZipFile
dataset = '/content/training.1600000.processed.noemoticon.csv.zip'

with ZipFile(dataset, 'r') as zip:
  zip.extractall()
  print('The dataset is extracted succesfully')

The dataset is extracted succesfully


# Importing necessary libraries including ML libraries.

In [180]:
import numpy as np
import pandas as pd
import  re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [181]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [187]:
print(stopwords.words('english')) # we can filter these words using stopwords command.

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

# Encoding the data from the zip file

In [176]:
twit_data = pd.read_csv(dataset,encoding = 'ISO-8859-1')

In [182]:
twit_data.shape

(1599999, 6)

In [188]:
twit_data.head(5)
# Here, the first row has considered as column names. But this is not true. So we need to add every column name additionally.

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


## Naming the columns and reading the dataset again.

In [184]:
columns_name = ['target','id','date','flag','user','text']
twit_data = pd.read_csv(dataset,names=columns_name, encoding = 'ISO-8859-1')

In [185]:
twit_data.shape # checking the shape is it 1600000 or not, as we have change the column name. Now the dataset becomes correct with 1600000 no of rows.

(1600000, 6)

In [186]:
twit_data.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [30]:
# counting the no of missing values in the dataset
twit_data.isnull().sum()

,0
target,0
id,0
date,0
flag,0
user,0
text,0


So, here is no null values. We can proceed with the dataset.

## Checking the distribution of the Target column

In [31]:
twit_data['target'].value_counts()

,count
target,
0,800000
4,800000


In place of '4' for positive class, it will be better to consider '1' for  understanding.

### Converting the level from '4' to '1'

In [32]:
from os import replace
twit_data.replace({'target':{4:1}}, inplace=True)

In [39]:
twit_data['target'].value_counts()

,count
target,
0,800000
1,800000


### Stemming

*   0 ---> Negetive

*   1 ---> Positive



In [33]:
port_stem = PorterStemmer() # For key text preprocessing

### Creating a function to convert 'text' column into a cleaned column where letters will be in lower case and no special charecters will be there. Using 'stopwords', we can filters the english words.

In [35]:
def stemming (content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)
  return stemmed_content

In [37]:
twit_data['stemmed_content'] = twit_data['text'].apply(stemming)

In [38]:
twit_data.head()

,target,id,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


## Seperating the data and type levels.

In [119]:
X = twit_data['stemmed_content'].values
Y = twit_data['target'].values

In [120]:
print(X)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


## Train, Test split ( 80% : 20% or 70% : 30% )

In [135]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.3, stratify = Y, random_state = 2)

In [136]:
print(X.shape,X_train.shape,X_test.shape)

(1600000,) (1120000,) (480000,)


In [137]:
print(X_train)

['finish read book second time'
 'tommygun back action lunch today best watchout bday boy take'
 'kata shall listen later' ... 'jedbramwel thank teddytuesday'
 'mtv movi award hope rob win realli care twilight win best movi rob win'
 'love wake folger bad voic deeper']


In [138]:
print(X_test)

['gylesoneshow sure find think may dead sorri break way'
 'final updat turn taken hr' 'rain' ...
 'soon go sligo coupl day bankholiday tomorrow see ya'
 'piaaguirr know right grabe manira' 'feelin empti lt']


Converting Vector values into numeric values. Tfidfvectorizer( ) converts a collection of raw text documents into a matrix of TF-IDF features.

## Transforming the text into matrix or numeric structures.

In [139]:
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)


## These values are converted into numeric values.

In [140]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 8272806 stored elements and shape (1120000, 422739)>
  Coords	Values
  (0, 121348)	0.44599875194306554
  (0, 306319)	0.4371164314704369
  (0, 43969)	0.4748653389926461
  (0, 328004)	0.5314491135690684
  (0, 374778)	0.31949818170660904
  (1, 377777)	0.5390797727604117
  (1, 28528)	0.16997429420382654
  (1, 2941)	0.3329517305729672
  (1, 224362)	0.2471619420602169
  (1, 376966)	0.16230856596272636
  (1, 36162)	0.21760975021723117
  (1, 400139)	0.5008486447803013
  (1, 32222)	0.2885646935506797
  (1, 45367)	0.24454930105105105
  (1, 360908)	0.20232270128356877
  (2, 192775)	0.7270515839609044
  (2, 331146)	0.4577243030925928
  (2, 217307)	0.3586212341678004
  (2, 209149)	0.3650688524406097
  (3, 175917)	0.5797213273864051
  (3, 139076)	0.25759117714273316
  (3, 366377)	0.4018889828996008
  (3, 344001)	0.3565976148233519
  (3, 114004)	0.3165169852577187
  (3, 381278)	0.3044716859774938
  :	:
  (1119996, 41689)	0.2988684615709349

In [141]:
print(X_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3425403 stored elements and shape (480000, 422739)>
  Coords	Values
  (0, 46696)	0.3105296872456904
  (0, 86927)	0.33413484518869224
  (0, 121198)	0.2668513552052889
  (0, 143634)	0.6303950980554028
  (0, 235076)	0.29762268984383516
  (0, 345122)	0.24201085278417414
  (0, 357350)	0.2678444483614686
  (0, 372248)	0.2111379764309065
  (0, 400455)	0.24464092290138587
  (1, 121102)	0.36962717098251946
  (1, 156589)	0.48451365961749343
  (1, 360939)	0.5151309759331816
  (1, 383626)	0.43305234092570793
  (1, 390597)	0.41919925544830766
  (2, 303665)	1.0
  (3, 28528)	0.2635724776921008
  (3, 39958)	0.6085945392762492
  (3, 72122)	0.26777644261468114
  (3, 156926)	0.2493726437719803
  (3, 281474)	0.6528792362184421
  (4, 180954)	0.5598634315653229
  (4, 189400)	0.463921117899177
  (4, 215156)	0.21842711142276236
  (4, 220281)	0.3037998272593176
  (4, 220487)	0.25816571428786494
  :	:
  (479996, 81006)	0.643199214486612
  (479996, 85

## Training the Machine learning model focused on Logistic Regression.

In [142]:
model = LogisticRegression()

In [143]:
model.fit(X_train,Y_train)

LogisticRegression()

## Accurecy score

In [144]:
X_train_prediction = model.predict(X_train)
training_accurecy = accuracy_score(X_train_prediction,Y_train)

In [145]:
print("For training data, the accuracy score is:",training_accurecy)

For training data, the accuracy score is: 0.8039133928571428


In [146]:
# Accurecy score
X_test_prediction = model.predict(X_test)
testing_accurecy = accuracy_score(X_test_prediction,Y_test)

In [147]:
print("For training data, the accuracy score is:",testing_accurecy)

For training data, the accuracy score is: 0.77708125


Here we can see that the Testing dataset's accurecy (80.39%) is closer to the training data det's accurecy (77.70%). So our model can classify the predicted value correctly with 77.70% of chance.

## Saving the training ML model

In [149]:
import pickle
file = 'trained_model.sav'
pickle.dump(model,open(file,'wb')) # for saving

USing the saved model for new data.

In [150]:
load_model = pickle.load(open('/content/trained_model.sav','rb')) # for reading

## Testing on different datasets.

In [171]:
X_test_new = X_test[100]
print("Actual class:",Y_test[100])
print("Predicted class:",load_model.predict(X_test_new)[0])
if(load_model.predict(X_test_new)[0] == 0):
  print("Predicted class is NEGETIVE")
else:
  print("Predicted class is POSITIVE")

Actual class: 1
Predicted class: 1
Predicted class is POSITIVE


From the above output we can conclude that

In [167]:
X_test_new = X_test[11158]
print("Actual class:",Y_test[11158])
print("Predicted class:",load_model.predict(X_test_new)[0]) # the model is predicting correctly for other data also.
if(load_model.predict(X_test_new)[0] == 0):
  print("Predicted class is NEGETIVE")
else:
  print("Predicted class is POSITIVE")

Actual class: 0
Predicted class: 0
Predicted class is NEGETIVE
